# News Dataset Generation

This notebook outlines the data preparation process for the `news_nytimes_lda.npz` file, which is used by the `NewsDataset` simulator in `pgmpy`. 

The dataset simulates a media consumer's reading experience of news articles under different viewing devices (desktop vs. mobile), implementing the benchmark introduced by Johansson et al. (2016).

The process includes:
1. Downloading the NY Times bag-of-words corpus from the UCI Machine Learning Repository (300,000 articles).
2. Parsing the data into memory-efficient sparse matrices.
3. Fitting Latent Dirichlet Allocation (LDA) models for $K=50$ and $K=200$ topics.
4. Extracting the vocabulary and visualizing the learned topics.
5. Saving the required arrays into an `.npz` format for hosting.

*Note: Generating this dataset requires fitting LDA on 300,000 documents. Running this notebook sequentially requires approximately 4-8 GB of RAM and may take 30-60 minutes depending on hardware.*


In [ ]:
import gzip
import logging
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np
from scipy import sparse
from sklearn.decomposition import LatentDirichletAllocation

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# Constants
UCI_BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/bag-of-words"
DOCWORD_URL = f"{UCI_BASE}/docword.nytimes.txt.gz"
VOCAB_URL = f"{UCI_BASE}/vocab.nytimes.txt"
CACHE_DIR = Path("_cache_news_prep")

SEED = 42
OUTPUT_FILE = "news_nytimes_lda.npz" 


## 1. Download Raw Data
Fetch the raw gzipped word counts and vocabulary list from the UCI Machine Learning Repository.


In [ ]:
def download_file(url: str, dest: Path) -> Path:
    if dest.exists():
        log.info(f"Using cached: {dest}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    log.info(f"Downloading {url} ...")
    try:
        urllib.request.urlretrieve(url, dest)
    except urllib.error.URLError as e:
        if "CERTIFICATE_VERIFY_FAILED" in str(e):
            import ssl
            log.warning("SSL verification failed, retrying with unverified context ...")
            ctx = ssl.create_default_context()
            ctx.check_hostname = False
            ctx.verify_mode = ssl.CERT_NONE
            opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
            with opener.open(url) as response, open(dest, "wb") as out_file:
                import shutil
                shutil.copyfileobj(response, out_file)
        else:
            raise
    log.info(f"Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest

docword_path = download_file(DOCWORD_URL, CACHE_DIR / "docword.nytimes.txt.gz")
vocab_path = download_file(VOCAB_URL, CACHE_DIR / "vocab.nytimes.txt")


## 2. Parsing the Dataset
The UCI docword format is parsed directly into a `scipy.sparse.csr_matrix` for memory efficiency.


In [ ]:
def load_vocabulary(vocab_path: Path) -> np.ndarray:
    log.info("Loading vocabulary ...")
    with open(vocab_path) as f:
        words = [line.strip() for line in f if line.strip()]
    log.info(f"Vocabulary size: {len(words)}")
    return np.array(words)

def load_docword_sparse(docword_path: Path) -> sparse.csr_matrix:
    log.info(f"Parsing sparse matrix from {docword_path} ...")
    t0 = time.time()
    open_fn = gzip.open if str(docword_path).endswith(".gz") else open
    
    with open_fn(docword_path, "rt") as f:
        D = int(f.readline().strip())
        W = int(f.readline().strip())
        NNZ = int(f.readline().strip())
        log.info(f"  D={D:,}, W={W:,}, NNZ={NNZ:,}")
        
        rows = np.empty(NNZ, dtype=np.int32)
        cols = np.empty(NNZ, dtype=np.int32)
        vals = np.empty(NNZ, dtype=np.int32)
        
        for i, line in enumerate(f):
            parts = line.split()
            rows[i] = int(parts[0]) - 1
            cols[i] = int(parts[1]) - 1
            vals[i] = int(parts[2])
            if (i + 1) % 20_000_000 == 0:
                log.info(f"  ... parsed {i + 1:,}/{NNZ:,} entries")
                
    mat = sparse.coo_matrix((vals, (rows, cols)), shape=(D, W)).tocsr()
    elapsed = time.time() - t0
    log.info(f"  Sparse matrix shape: {mat.shape}, nnz: {mat.nnz:,}, parsed in {elapsed:.1f}s")
    return mat

full_vocab = load_vocabulary(vocab_path)
word_counts_full = load_docword_sparse(docword_path)


## 3. Topic Modeling (LDA)
Latent Dirichlet Allocation (LDA) models are fit to the dataset.
*   **$K=50$**: The standard hyperparameter used in the original benchmarking paper.
*   **$K=200$**: Pre-computed to allow `pgmpy` to use agglomerative clustering for dynamically merging topics on the fly (supporting $K \le 200$).


In [ ]:
def fit_lda(word_counts: sparse.csr_matrix, n_components: int, seed: int) -> tuple:
    log.info(f"Fitting LDA with K={n_components} (batch mode, random_state={seed}) ...")
    t0 = time.time()
    lda = LatentDirichletAllocation(
        n_components=n_components,
        learning_method="batch",
        random_state=seed,
        n_jobs=1,
        verbose=10,
    )
    topic_dists = lda.fit_transform(word_counts)
    elapsed = time.time() - t0
    log.info(f"  LDA K={n_components} fit in {elapsed:.1f}s ({elapsed / 60:.1f} min)")
    return topic_dists, lda.components_

topic_dists_k50, lda_components_k50 = fit_lda(word_counts_full, n_components=50, seed=SEED)
topic_dists_k200, lda_components_k200 = fit_lda(word_counts_full, n_components=200, seed=SEED)


## 4. Vocabulary Selection
The covariate vocabulary is constructed by taking the union of the top 100 words from each of the 50 baseline topics.


In [ ]:
def select_vocabulary(lda_components: np.ndarray, top_n_words: int = 100) -> np.ndarray:
    top_words_per_topic = np.argsort(lda_components, axis=1)[:, -top_n_words:]
    vocab_indices = np.unique(top_words_per_topic.flatten())
    log.info(
        f"  Selected {len(vocab_indices)} unique words from top-{top_n_words} across {lda_components.shape[0]} topics"
    )
    return vocab_indices

vocab_indices = select_vocabulary(lda_components_k50, top_n_words=100)
vocab_3477 = full_vocab[vocab_indices]
word_counts_3477 = word_counts_full[:, vocab_indices]

log.info(f"Final Vocabulary shape: {vocab_3477.shape}")
log.info(f"Subset word counts shape: {word_counts_3477.shape}")


## 5. Visualizing the Learned Topics
Display the top words for the first 10 topics to verify semantic coherence.


In [ ]:
def display_topics(components, feature_names, no_top_words, n_topics=10):
    for topic_idx, topic in enumerate(components[:n_topics]):
        print(f"Topic {topic_idx + 1}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))
        print()

display_topics(lda_components_k50, full_vocab, 15, n_topics=10)


## 6. Saving the NPZ File
The data is packaged into a compressed `.npz` file. Since `np.savez_compressed` does not natively support `scipy.sparse` matrices, they are deconstructed into their underlying CSR arrays (`data`, `indices`, `indptr`).


In [ ]:
# Convert to float32 for storage efficiency
topic_dists_k50 = topic_dists_k50.astype(np.float32)
topic_dists_k200 = topic_dists_k200.astype(np.float32)
lda_components_k50 = lda_components_k50.astype(np.float32)
lda_components_k200 = lda_components_k200.astype(np.float32)

log.info(f"Saving to {OUTPUT_FILE} ...")
t0 = time.time()

np.savez_compressed(
    OUTPUT_FILE,
    # Precomputed topic distributions
    topic_distributions_k50=topic_dists_k50,
    topic_distributions_k200=topic_dists_k200,
    # LDA components
    lda_components_k50=lda_components_k50,
    lda_components_k200=lda_components_k200,
    # Precomputed vocabulary
    vocab_3477=vocab_3477,
    # Word counts at defaults
    word_counts_3477_data=word_counts_3477.data if sparse.issparse(word_counts_3477) else word_counts_3477,
    word_counts_3477_indices=word_counts_3477.indices if sparse.issparse(word_counts_3477) else np.array([]),
    word_counts_3477_indptr=word_counts_3477.indptr if sparse.issparse(word_counts_3477) else np.array([]),
    word_counts_3477_shape=np.array(word_counts_3477.shape),
    # Full word counts
    word_counts_full_data=word_counts_full.data,
    word_counts_full_indices=word_counts_full.indices,
    word_counts_full_indptr=word_counts_full.indptr,
    word_counts_full_shape=np.array(word_counts_full.shape),
    # Full vocabulary
    full_vocab=full_vocab,
)

file_size = os.path.getsize(OUTPUT_FILE)
elapsed = time.time() - t0
log.info(f"Saved {OUTPUT_FILE} ({file_size / 1e6:.1f} MB) in {elapsed:.1f}s")
